In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!mv kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# ============================================================
# UNSW-NB15 — Descarga Kaggle + Unificación + Limpieza + Normalización (1 script)
# VERSIÓN CORREGIDA (ver notas más abajo respecto a la original de TFM1):
#   1. Se recupera "Reconnaissance" como clase propia (antes se fusionaba
#      con "PortScan"). El sistema MODEXRE ya distingue ambas categorías
#      en su taxonomía (app/ocsf/mappers.py::ALLOWED_ATTACK_CATEGORIES),
#      por lo que fusionarlas aquí le ocultaba al modelo la posibilidad
#      de aprenderlas por separado.
#   2. Se recupera "Worms" como clase propia (antes caía a "Generic" por
#      no estar en ALLOWED/MAP).
#   3. Se CONSERVAN srcip/dstip/sport/dsport (antes se eliminaban). Son
#      imprescindibles para poder simular más adelante secuencias de
#      eventos por origen (necesarias para entrenar las variables de
#      agregación agg_distinct_dst_ports/agg_distinct_dst_hosts/
#      agg_events_in_window que usa model_v1 -- ver
#      backend/app/features/flow_aggregation.py del proyecto MODEXRE).
#   4. Cualquier clase no reconocida por ALLOWED/MAP ya NO se colapsa
#      a "Generic": se conserva con su propio nombre. Antes, cualquier
#      etiqueta inesperada se perdía mezclada dentro de "Generic".
# Salida: /content/UNSW_NB15_full_clean.csv
#   - attack_cat (normalizada, con Reconnaissance y Worms ya
#     recuperadas como clases propias, y cualquier clase nueva
#     conservada con su propio nombre)
#   - label binaria 0/1 en category (derivada SOLO de attack_cat)
#   - attack_cat aparece ANTES que label en el CSV final
#   - srcip/dstip/sport/dsport se conservan
# ============================================================

!pip install -q kaggle pandas numpy tqdm

import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ===================== CONFIG =====================
KAGGLE_DATASET = "mrwellsdavid/unsw-nb15"
ZIP_PATH = "/content/unsw-nb15.zip"
EXTRACT_DIR = "/content/unsw-nb15"
OUT_CSV = "/content/UNSW_NB15_full_clean_v2.csv"
# ==================================================

# ============================================================
# NORMALIZADOR attack_cat (MISMA TAXONOMÍA QUE KITSUNE/CICIDS,
# CON Reconnaissance Y Worms YA RECUPERADAS COMO CLASES PROPIAS)
# ============================================================
ALLOWED = {
    "Normal",
    "Fuzzers", "Exploits", "DoS", "Reconnaissance", "Generic", "Analysis",
    "Shellcode", "Backdoors", "DDoS", "PortScan", "MitM", "BruteForce",
    "Worms",
}

MAP = {
    "Benign": "Normal",
    "BENIGN": "Normal",
    "normal": "Normal",
    "benign": "Normal",

    # Reconnaissance ya NO se fusiona con PortScan: son categorías
    # distintas en la taxonomía de MODEXRE, y fusionarlas le impedía al
    # modelo aprender a distinguir el patrón de "barrido de red /
    # descubrimiento de hosts" (Reconnaissance) del escaneo de puertos
    # vertical clásico (PortScan).
    "Port Scan": "PortScan",
    "Portscan": "PortScan",
    "portscan": "PortScan",
    "PortScan": "PortScan",
    "Reconnaissance": "Reconnaissance",
    "reconnaissance": "Reconnaissance",

    "bruteforce": "BruteForce",
    "BruteForce": "BruteForce",
    "brute force": "BruteForce",
    "Brute Force": "BruteForce",

    "Ddos": "DDoS",
    "ddos": "DDoS",
    "DDoS": "DDoS",

    "Dos": "DoS",
    "dos": "DoS",
    "DoS": "DoS",

    "MITM": "MitM",
    "mitm": "MitM",
    "MitM": "MitM",

    "Worms": "Worms",
    "worms": "Worms",
    "Worm": "Worms",
    "worm": "Worms",

    # UNSW-NB15 trae la etiqueta nativa en singular ("Backdoor"), pero
    # la taxonomía de MODEXRE usa el plural ("Backdoors"). Sin este
    # mapeo, ambas quedarían como dos clases distintas en el
    # entrenamiento (visto en ejecución real: 'Backdoor': 30000 y
    # 'Backdoors': 30000 por separado), diluyendo artificialmente la
    # señal de esa categoría entre dos etiquetas que son la misma.
    "Backdoor": "Backdoors",
    "backdoor": "Backdoors",
    "Backdoors": "Backdoors",
    "backdoors": "Backdoors",

    # clave:
    "Attack": "Generic",
    "attack": "Generic",
    "Unknown": "Generic",
    "": "Normal",
    "None": "Normal",
    "nan": "Normal",
    "NaN": "Normal",
}

def normalize_cols(cols):
    return [str(c).strip().replace(" ", "_").replace("/", "_").replace("-", "_").lower() for c in cols]

def normalize_attack_cat_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.replace(MAP)
    # normal exacto
    s = s.apply(lambda x: "Normal" if str(x).strip().lower() == "normal" else x)
    # Vacío/NaN -> Normal. Cualquier OTRA clase, esté o no en ALLOWED,
    # se conserva TAL CUAL (título capitalizado por consistencia): no
    # se colapsa a "Generic". Antes, cualquier valor fuera de ALLOWED
    # se perdía dentro de "Generic", ocultando posibles categorías
    # nuevas o mal escritas que en realidad merecían su propio nombre.
    s = s.apply(lambda x: "Normal" if str(x).strip() == "" else str(x).strip())
    return s

def enforce_attackcat_label(df: pd.DataFrame) -> pd.DataFrame:
    """
    CHECK DURO ÚNICO:
      - attack_cat == Normal => label=0
      - attack_cat != Normal => label=1
    """
    df = df.copy()
    if "attack_cat" not in df.columns:
        raise ValueError("Falta attack_cat.")
    df["attack_cat"] = normalize_attack_cat_series(df["attack_cat"])
    df["label"] = (df["attack_cat"] != "Normal").astype(int).astype("category")
    return df

# ===================== 1) Descargar y descomprimir =====================
print(f">> Descargando dataset Kaggle: {KAGGLE_DATASET}")
t0 = time.time()
!kaggle datasets download -d $KAGGLE_DATASET -p /content/ -w
print(f"[OK] Descargado en {time.time()-t0:.1f}s")

print(">> Descomprimiendo…")
t1 = time.time()
!unzip -o /content/unsw-nb15.zip -d /content/unsw-nb15 > /dev/null
print(f"[OK] Extraído en {(time.time()-t1)/60:.2f} min → {EXTRACT_DIR}")

# ===================== 2) Localizar partes + features =====================
base = Path(EXTRACT_DIR)
parts = [base / f"UNSW-NB15_{i}.csv" for i in range(1, 5)]
features_path = base / "NUSW-NB15_features.csv"  # suele venir con ese nombre

for p in parts:
    if not p.exists():
        raise FileNotFoundError(f"No existe: {p}")
if not features_path.exists():
    raise FileNotFoundError(f"No existe: {features_path}")

# ===================== 3) Leer diccionario features =====================
t2 = time.time()
feat = pd.read_csv(features_path, encoding="latin1")
name_col = None
for cand in ["Name","name","Feature","feature","Attribute","attribute"]:
    if cand in feat.columns:
        name_col = cand
        break
if name_col is None:
    if feat.shape[1] == 1:
        name_col = feat.columns[0]
    else:
        raise ValueError("No encuentro la columna con nombres en NUSW-NB15_features.csv")

feature_names = feat[name_col].astype(str).str.strip().tolist()
print(f"[OK] Features leídas en {time.time()-t2:.1f}s | n_features={len(feature_names)}")

# ===================== 4) Leer fragmentos con progreso =====================
dfs = []
t3 = time.time()
for p in tqdm(parts, desc="Leyendo fragmentos UNSW (1/4..4/4)", unit="fichero"):
    tmp = pd.read_csv(p, header=None, low_memory=False, encoding="latin1")

    if tmp.shape[1] == len(feature_names):
        tmp.columns = feature_names
    else:
        tmp2 = pd.read_csv(p, header=0, low_memory=False, encoding="latin1")
        if tmp2.shape[1] == len(feature_names):
            tmp2.columns = feature_names
            tmp = tmp2
        else:
            raise ValueError(
                f"Las columnas de {p.name} no coinciden con el diccionario de features: "
                f"{tmp.shape[1]} vs {len(feature_names)}"
            )

    dfs.append(tmp)
print(f"[OK] Fragmentos cargados en {(time.time()-t3)/60:.2f} min")

# ===================== 5) Unir + normalizar columnas =====================
t4 = time.time()
df = pd.concat(dfs, ignore_index=True)
del dfs
df.columns = normalize_cols(df.columns.tolist())
print(f"[OK] Concatenado en {time.time()-t4:.1f}s | shape={df.shape}")

# ===================== 6) Limpieza numérica mínima (sin dropna global) =====================
t5 = time.time()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
print(f"[OK] inf→NaN en {time.time()-t5:.1f}s | num_cols={len(num_cols)}")

t6 = time.time()
for c in tqdm(num_cols, desc="Imputando numéricas (mediana)", unit="col"):
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(0.0 if np.isnan(med) else med)
print(f"[OK] Imputación numérica en {(time.time()-t6)/60:.2f} min")

# ===================== 7) Asegurar attack_cat, normalizar a ALLOWED y CHECK DURO =====================
if "label" not in df.columns:
    raise ValueError("No encuentro 'label' en UNSW (revisa features).")

# Si faltase attack_cat (no debería), lo creamos desde label:
if "attack_cat" not in df.columns:
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int).clip(0, 1)
    df["attack_cat"] = np.where(df["label"] == 0, "Normal", "Generic")

# Normalizar y CHECK DURO único
df = enforce_attackcat_label(df)

# ===================== 8) IP/puertos: SE CONSERVAN =====================
# (antes se eliminaban aquí "por seguridad" -- ya no: son necesarias
# para poder construir escenarios de agregación por origen más
# adelante. Solo se normaliza el nombre por consistencia, no se borra
# ninguna columna.)
ip_port_cols_present = [c for c in ("srcip", "dstip", "sport", "dsport") if c in df.columns]
print(f"[INFO] Columnas de IP/puerto conservadas: {ip_port_cols_present}")

# ===================== 9) Reordenar columnas: attack_cat, label, IP/puerto, resto =====================
cols = df.columns.tolist()
for k in ["attack_cat", "label"] + ip_port_cols_present:
    if k in cols:
        cols.remove(k)
new_cols = ["attack_cat", "label"] + ip_port_cols_present + cols
df = df[new_cols]

# ===================== 10) Guardar CSV =====================
t7 = time.time()
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"[OK] Guardado en {time.time()-t7:.1f}s → {OUT_CSV}")
print("Shape final:", df.shape)

# ===================== 11) Verificaciones finales =====================
print("\nColumnas clave presentes:", [c for c in df.columns if c in ["attack_cat","label","proto","service","state","srcip","dstip","sport","dsport"]])

print("\nDistribución label (binaria):")
print(df["label"].astype(str).value_counts(normalize=True).round(3))

print("\nattack_cat (todas las clases, incluye Reconnaissance y Worms si están presentes en el dataset original):")
print(df["attack_cat"].value_counts())

print("\n[VERIFICACIÓN FINAL]")
print("Generic con label=0 →", int(((df["attack_cat"]=="Generic") & (df["label"].astype(int)==0)).sum()))
print("Normal con label=1  →", int(((df["attack_cat"]=="Normal")  & (df["label"].astype(int)==1)).sum()))
print("Reconnaissance detectada como clase propia →", int((df["attack_cat"]=="Reconnaissance").sum()), "filas")
print("Worms detectada como clase propia →", int((df["attack_cat"]=="Worms").sum()), "filas")

#df.head(5)

In [ ]:
# === UNSW-NB15 — Synthetic por cuotas (GaussianCopula) ===
# NOTA: ahora que srcip/dstip/sport/dsport sobreviven en el CSV real,
# el sintetizador SDV las tratará como columnas más (categóricas o
# numéricas según detecte). Esto es intencional: para el objetivo de
# entrenar escenarios de agregación, lo relevante es tener disponible
# la columna de IP en el fichero de salida, aunque el GaussianCopula
# por sí solo NO genera secuencias temporales coherentes por origen
# (eso requiere un paso de construcción de escenarios aparte, no una
# síntesis fila a fila como esta). Ver notebook/script de escenarios.
!pip install -q sdv tqdm pandas numpy

import os, math, gc, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

# ===================== CONFIG =====================
REAL_CSV  = "/content/UNSW_NB15_full_clean_v2.csv"
SYN_CSV   = "/content/synthetic_UNSW_NB15_ctgan_v2.csv"
META_JSON = "/content/synthetic_UNSW_NB15_ctgan_v2.json"

RANDOM_STATE = 42

NORMAL_N      = 200_000
ATTACK_N_EACH = 30_000

CHUNK_SIZE = 250_000

TRAIN_TOTAL_N         = 250_000
TRAIN_NORMAL_MAX      = 140_000
# Antes 35_000: con Reconnaissance/Worms recuperadas, algunas clases
# nativas tienen menos filas reales que este mínimo (p.ej. Worms tiene
# solo ~174 en todo el dataset). Se baja el mínimo exigido para no
# forzar un error, y se toma como aviso informativo si una clase no lo
# alcanza (ver aviso más abajo), en vez de fallar.
TRAIN_ATTACK_MIN_EACH = 150

DEFAULT_DISTRIBUTION = "gamma"
ENFORCE_MINMAX = False
# ================================================

def norm_cols(cols):
    return [str(c).strip().replace(" ", "_").replace("/", "_").replace("-", "_").lower() for c in cols]

def enforce_label_attackcat(df):
    df = df.copy()
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip()
    df["label"] = (df["attack_cat"].str.lower() != "normal").astype(int)
    return df

def approx_line_count(path: str) -> int:
    with open(path, "rb") as f:
        return max(sum(1 for _ in f) - 1, 0)

def append_csv(df_part, path):
    header = not os.path.exists(path)
    df_part.to_csv(path, index=False, mode="a", header=header)

# ===================== scan clases =====================
n_lines = approx_line_count(REAL_CSV)
n_chunks = max(1, int(math.ceil(n_lines / CHUNK_SIZE)))
cnt_attack = Counter()

reader = pd.read_csv(REAL_CSV, chunksize=CHUNK_SIZE, low_memory=False)
for ch in tqdm(reader, total=n_chunks, desc="Scan clases UNSW", unit="chunk"):
    ch.columns = norm_cols(ch.columns)
    cnt_attack.update(ch["attack_cat"].astype(str).str.strip().tolist())
    del ch
    gc.collect()

attack_cats = sorted([c for c in cnt_attack.keys() if str(c).strip().lower() != "normal"])
print("[INFO] attack_cat detectadas (sin Normal):", attack_cats)
for cat in attack_cats:
    if cnt_attack[cat] < TRAIN_ATTACK_MIN_EACH:
        print(f"[AVISO] La clase '{cat}' tiene solo {cnt_attack[cat]} filas reales "
              f"(por debajo del mínimo objetivo {TRAIN_ATTACK_MIN_EACH}). Se usarán "
              f"todas las disponibles, pero la clase quedará infrarrepresentada.")

# ===================== construir df_train =====================
rng = np.random.RandomState(RANDOM_STATE)
train_parts, seen = [], Counter()

reader = pd.read_csv(REAL_CSV, chunksize=CHUNK_SIZE, low_memory=False)
for ch in tqdm(reader, total=n_chunks, desc="df_train UNSW", unit="chunk"):
    ch.columns = norm_cols(ch.columns)
    ch = enforce_label_attackcat(ch)

    for cat in attack_cats:
        need = TRAIN_ATTACK_MIN_EACH - seen[cat]
        if need <= 0:
            continue
        sub = ch[ch["attack_cat"] == cat]
        if len(sub) == 0:
            continue
        take = min(need, len(sub))
        train_parts.append(sub.sample(n=take, random_state=int(rng.randint(0, 1e9))))
        seen[cat] += take

    needN = TRAIN_NORMAL_MAX - seen["Normal"]
    if needN > 0:
        subN = ch[ch["attack_cat"].str.lower() == "normal"]
        if len(subN) > 0:
            takeN = min(needN, len(subN))
            train_parts.append(subN.sample(n=takeN, random_state=int(rng.randint(0, 1e9))))
            seen["Normal"] += takeN

    del ch
    gc.collect()

    if sum(seen.values()) >= TRAIN_TOTAL_N:
        break

df_train = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE)
df_train = df_train.head(TRAIN_TOTAL_N).reset_index(drop=True)

# ===================== Exportar muestra manejable para MODEXRE =====================
# df_train es una muestra estratificada y balanceada (no el CSV
# completo de 2,5M de filas): es la que hay que subir a la pestaña
# Laboratorio de MODEXRE para entrenar/certificar, no el
# UNSW_NB15_full_clean.csv completo (que puede pesar varios cientos de
# MB y superar el límite de subida de Streamlit, además de ser
# innecesario cargarlo entero en memoria para entrenar).
TRAIN_SAMPLE_CSV = "/content/UNSW_NB15_train_sample_v2.csv"
df_train.to_csv(TRAIN_SAMPLE_CSV, index=False, encoding="utf-8")
print(f"[OK] Muestra de entrenamiento exportada → {TRAIN_SAMPLE_CSV}")
print(f"     Shape: {df_train.shape} (frente a las {len(df):,} filas del CSV completo)")
print(f"     Peso aproximado: {os.path.getsize(TRAIN_SAMPLE_CSV) / (1024*1024):.1f} MB")
print(f"     Distribución de attack_cat en la muestra:")
print(df_train["attack_cat"].value_counts())

# ===================== fit SDV =====================
df_sdv = df_train.copy()
for c in df_sdv.columns:
    if str(df_sdv[c].dtype) == "category":
        df_sdv[c] = df_sdv[c].astype("object")

df_sdv["attack_cat"] = df_sdv["attack_cat"].astype(str)
df_sdv["label"] = df_sdv["label"].astype(str)

metadata = Metadata.detect_from_dataframe(df_sdv)

# srcip/dstip: SDV las detectaría por defecto como texto libre/PII
# genérico. Las marcamos explícitamente como categóricas para que el
# sintetizador muestree de los valores reales vistos en vez de generar
# texto arbitrario sin sentido de red.
for ip_col in ("srcip", "dstip"):
    if ip_col in df_sdv.columns:
        metadata.update_column(column_name=ip_col, sdtype="categorical")

metadata.save_to_json(META_JSON)

synth = GaussianCopulaSynthesizer(
    metadata,
    default_distribution=DEFAULT_DISTRIBUTION,
    enforce_min_max_values=ENFORCE_MINMAX
)

print("[INFO] Entrenando sintetizador UNSW...")
synth.fit(df_sdv)

# ===================== generar cuotas =====================
if os.path.exists(SYN_CSV):
    os.remove(SYN_CSV)

synN = synth.sample(num_rows=NORMAL_N)
synN.columns = norm_cols(synN.columns)
synN = enforce_label_attackcat(synN)
synN["attack_cat"] = "Normal"
synN["label"] = 0
append_csv(synN, SYN_CSV)
del synN
gc.collect()

for cat in tqdm(attack_cats, desc="Generando ataques UNSW", unit="clase"):
    synA = synth.sample(num_rows=ATTACK_N_EACH)
    synA.columns = norm_cols(synA.columns)
    synA = enforce_label_attackcat(synA)
    synA["attack_cat"] = str(cat)
    synA["label"] = 1
    append_csv(synA, SYN_CSV)
    del synA
    gc.collect()

print("[OK] Sintético guardado:", SYN_CSV)

# validación
cnt_syn = Counter()
for ch in pd.read_csv(SYN_CSV, chunksize=200_000):
    cnt_syn.update(ch["attack_cat"].astype(str).str.strip().tolist())
print("[SYN] attack_cat:", dict(cnt_syn))

In [ ]:
print("[REAL] attack_cat:", pd.read_csv("/content/UNSW_NB15_full_clean.csv", low_memory=False)["attack_cat"].value_counts().head(30))
print("[SYN]  attack_cat:", pd.read_csv("/content/synthetic_UNSW_NB15_ctgan.csv", low_memory=False)["attack_cat"].value_counts().head(30))

In [ ]:
from google.colab import files
files.download("/content/UNSW_NB15_full_clean_v2.csv")
files.download("/content/UNSW_NB15_train_sample_v2.csv")
files.download("/content/synthetic_UNSW_NB15_ctgan_v2.csv")
files.download("/content/synthetic_UNSW_NB15_ctgan_v2.json")